# Module 02 — Supervised Fine-Tuning (SFT) with LoRA

**Goal**: Fine-tune a language model to follow instructions using TRL's `SFTTrainer` and LoRA.

**What you will build**:
1. Format an instruction dataset with chat templates
2. Configure a LoRA adapter (< 1% trainable parameters)
3. Run `SFTTrainer` for 100 steps
4. Compare outputs before and after training

**Hardware**: A100 80GB (runs in ~4 min). Adjust `MAX_STEPS` and `MODEL_ID` for smaller GPUs.

In [ ]:
# Cell 1 — Install
!pip install -q transformers datasets trl peft accelerate bitsandbytes torch

## 1. What is SFT?

The first post-training stage.

**Input**: A dataset of (instruction, response) pairs — curated human demonstrations.

**Training**: The model is trained to *reproduce* those responses given the instructions, using standard cross-entropy loss on the response tokens only.

**Output**: A model that follows the format and style of the demonstrations.

```
Loss = -Σ log P(response_token_i | instruction, response_tokens_<i)
       ↑ only over response tokens, NOT prompt tokens
```

In [ ]:
# Cell 2 — Imports & config
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import Dataset
from trl import SFTTrainer, SFTConfig
from peft import LoraConfig, TaskType

MODEL_ID  = "HuggingFaceTB/SmolLM2-135M-Instruct"
DEVICE    = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE     = torch.bfloat16
MAX_STEPS = 100   # increase to 500-1000 for real training
print(f"Device: {DEVICE}")

## 2. Create an instruction dataset

We build a tiny dataset inline. In a real project, replace this with a HuggingFace dataset:
```python
from datasets import load_dataset
ds = load_dataset("HuggingFaceH4/ultrachat_200k", split="train_sft[:5000]")
```

In [ ]:
# Cell 3 — Build inline instruction dataset
RAW_DATA = [
    {"instruction": "What is machine learning?",
     "response": "Machine learning is a branch of AI where models learn patterns from data rather than being explicitly programmed. Given examples of inputs and outputs, the model learns a mapping between them."},
    {"instruction": "Explain gradient descent in one paragraph.",
     "response": "Gradient descent is an optimisation algorithm that minimises a loss function by iteratively moving in the direction of the steepest downhill gradient. At each step we compute the gradient of the loss with respect to the model parameters and subtract a small fraction of it (the learning rate) from those parameters. Repeating this process eventually converges to a local minimum."},
    {"instruction": "What is the difference between a list and a tuple in Python?",
     "response": "A list is mutable — you can add, remove or change elements after creation. A tuple is immutable — once created, its contents cannot be changed. Tuples are slightly faster and are used when data should not be modified, such as function return values or dictionary keys."},
    {"instruction": "Write a Python function that returns the Fibonacci sequence up to n.",
     "response": "def fibonacci(n):\n    seq = [0, 1]\n    while seq[-1] + seq[-2] <= n:\n        seq.append(seq[-1] + seq[-2])\n    return seq"},
    {"instruction": "What does 'overfitting' mean in ML?",
     "response": "Overfitting happens when a model learns the training data so well — including its noise — that it performs poorly on unseen data. The model has memorised rather than generalised. Signs include low training loss but high validation loss. Fixes include regularisation, dropout, more data, or a simpler model."},
    {"instruction": "Explain the attention mechanism briefly.",
     "response": "Attention allows a model to focus on different parts of the input when producing each output token. For each position, it computes a query vector, then measures compatibility with key vectors from all positions via dot products. These scores become weights (after softmax) over value vectors, producing a weighted sum. This lets the model dynamically decide what context is relevant at each step."},
    {"instruction": "What is a learning rate?",
     "response": "The learning rate controls how large a step the optimiser takes when updating model parameters. Too high and the loss oscillates or diverges; too low and training is slow or gets stuck. Common values are 1e-4 to 3e-4 for fine-tuning and 1e-3 to 1e-2 for training from scratch."},
    {"instruction": "What is the difference between parameters and hyperparameters?",
     "response": "Parameters are the values a model learns during training — the weights and biases. Hyperparameters are settings you choose before training that control the learning process, such as learning rate, batch size, number of layers, and dropout rate."},
    {"instruction": "What is tokenization?",
     "response": "Tokenization converts raw text into a sequence of integers (token IDs) that a model can process. Modern LLMs use byte-pair encoding (BPE), which learns a vocabulary of subword units. The word 'unbelievable' might be split into ['un', 'believ', 'able'] and mapped to IDs like [1249, 8733, 603]."},
    {"instruction": "What is LoRA?",
     "response": "LoRA (Low-Rank Adaptation) is a parameter-efficient fine-tuning method. Instead of updating all model weights, LoRA freezes the original weights and adds small trainable rank-decomposition matrices to specific layers. If a weight matrix W is d×d, LoRA adds B×A where B is d×r and A is r×d, with r << d. This reduces trainable parameters by 100-10000× while retaining most of the fine-tuning quality."},
]

# Duplicate to get more training examples
RAW_DATA = RAW_DATA * 5  # 50 examples total

# Format using the chat template
tokenizer_for_template = AutoTokenizer.from_pretrained(MODEL_ID)

def format_example(row):
    messages = [
        {"role": "user",      "content": row["instruction"]},
        {"role": "assistant", "content": row["response"]},
    ]
    return {"text": tokenizer_for_template.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )}

raw_ds  = Dataset.from_list(RAW_DATA)
train_ds = raw_ds.map(format_example)

print(f"Dataset size: {len(train_ds)}")
print("\nExample formatted text:")
print(train_ds[0]["text"][:400])

## 3. Load model

We start from the instruct model as our base (this is called "continued SFT").
In practice you can also start from the raw base model.

In [ ]:
# Cell 4 — Load model and tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token  # SmolLM2 has no separate pad token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=DTYPE,
    device_map=DEVICE,
)

# Quick baseline — what does the model say BEFORE training?
def quick_infer(model, tok, question, max_new=100):
    messages = [{"role": "user", "content": question}]
    prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        ids = model.generate(**inputs, max_new_tokens=max_new,
                             temperature=0.7, do_sample=True,
                             pad_token_id=tok.eos_token_id)
    return tok.decode(ids[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)

TEST_Q = "What is LoRA and why is it useful?"
print("BEFORE training:")
print(quick_infer(model, tokenizer, TEST_Q))

## 4. Configure LoRA

Instead of updating all 135M parameters we add a tiny adapter: ~0.8% of the weights.

In [ ]:
# Cell 5 — LoRA configuration
lora_config = LoraConfig(
    r=8,               # rank — higher = more capacity, more memory
    lora_alpha=16,     # scaling factor (typically 2×r)
    target_modules=[   # which weight matrices to adapt
        "q_proj", "k_proj", "v_proj", "o_proj",   # attention
        "gate_proj", "up_proj", "down_proj",        # MLP
    ],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

# How many parameters will we actually train?
from peft import get_peft_model
peft_model = get_peft_model(model, lora_config)
peft_model.print_trainable_parameters()
# Expected: ~0.8% of 135M ≈ ~1M trainable params

## 5. Train with SFTTrainer

In [ ]:
# Cell 6 — SFT training
sft_config = SFTConfig(
    output_dir="/tmp/sft_smollm",
    max_steps=MAX_STEPS,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,    # effective batch = 8
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=10,
    logging_steps=10,
    save_steps=50,
    bf16=True,
    max_seq_length=512,
    dataset_text_field="text",
    report_to="none",
)

trainer = SFTTrainer(
    model=peft_model,
    args=sft_config,
    train_dataset=train_ds,
    processing_class=tokenizer,
)

print("Starting SFT training...")
trainer.train()
print("\nTraining complete!")

## 6. Compare before vs after

In [ ]:
# Cell 7 — After-training inference
# Merge LoRA weights back into the base model for clean inference
from peft import PeftModel
merged = peft_model.merge_and_unload()
merged.eval()

print("TEST QUESTION:", TEST_Q)
print()
print("AFTER SFT training:")
print(quick_infer(merged, tokenizer, TEST_Q))

In [ ]:
# Cell 8 — Loss curve
import matplotlib.pyplot as plt

log_history = trainer.state.log_history
steps  = [x["step"]        for x in log_history if "loss" in x]
losses = [x["loss"]        for x in log_history if "loss" in x]

plt.figure(figsize=(8, 4))
plt.plot(steps, losses, color="#2a9d8f", linewidth=2)
plt.title("SFT Training Loss")
plt.xlabel("Step")
plt.ylabel("Cross-entropy loss")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("/tmp/sft_loss.png", dpi=120)
plt.show()
print("Loss started at:", f"{losses[0]:.3f}", " → ended at:", f"{losses[-1]:.3f}")

## ✅ Summary

| Concept | What it means |
|---------|--------------|
| **SFT dataset** | (instruction, response) pairs formatted with chat template |
| **Loss** | Cross-entropy on *response tokens only* — prompt tokens are masked |
| **LoRA** | Adds < 1% trainable parameters; frozen base weights stay unchanged |
| **SFTTrainer** | TRL wrapper that handles masking, formatting, and the training loop |

**Next**: Module 03 trains a *reward model* that scores responses — the prerequisite for RLHF.

> 💡 For production SFT, replace the inline dataset with:
> ```python
> ds = load_dataset("HuggingFaceH4/ultrachat_200k", split="train_sft[:20000]")
> ```